# OCR Extraction Pipeline — Setup & Test

Full litmus test for the extraction pipeline. This notebook will:
1. Check GPU and shared models
2. Start vLLM
3. Load a sample document and check digital vs scanned
4. Run extraction
5. Inspect JSON output and experiment with formats
6. Process all pages
7. Launch Streamlit app

**Before starting:** Upload your sample PDFs/TIFFs to `/ocr/` using
Jupyter's file upload button.

## 1. Check GPU and shared models

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv,noheader

In [ ]:
from pathlib import Path

models_dir = Path("/models/.cache/huggingface")
if models_dir.exists():
    model_dirs = [d.name for d in models_dir.iterdir() if d.name.startswith("models--")]
    print(f"Shared models PVC mounted. {len(model_dirs)} model(s):")
    for m in sorted(model_dirs):
        print(f"  {m}")
    
    # Check for Qwen2.5-VL specifically
    has_vlm = any("Qwen2.5-VL" in d for d in model_dirs)
    if has_vlm:
        print("\nQwen2.5-VL found.")
    else:
        print("\nWARNING: Qwen2.5-VL not found on PVC!")
        print("Download it first: python /models/provision_shared_models.py download Qwen/Qwen2.5-VL-7B-Instruct")
else:
    print("WARNING: /models/.cache/huggingface not found.")
    print("Is the shared-models data volume attached?")

## 2. Start vLLM

This launches vLLM as a background process. Takes ~1-2 min to load the model.

In [ ]:
import subprocess
import time
import os

os.environ["HF_HOME"] = "/models/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/models/.cache/huggingface"
os.environ["HF_HUB_OFFLINE"] = "1"

vllm_proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", "Qwen/Qwen2.5-VL-7B-Instruct",
     "--dtype", "auto",
     "--max-model-len", "8192",
     "--limit-mm-per-prompt", "image=1"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"vLLM starting (PID {vllm_proc.pid})...")
print("Wait for the next cell to confirm it's ready.")

In [ ]:
import httpx

LLM_BASE_URL = "http://localhost:8000/v1"

# Poll until vLLM is ready (up to 5 min)
for i in range(60):
    try:
        resp = httpx.get(f"{LLM_BASE_URL}/models", timeout=5.0)
        models = resp.json()
        MODEL_ID = models["data"][0]["id"]
        print(f"vLLM ready! Model: {MODEL_ID}")
        break
    except Exception:
        if i % 6 == 0:
            print(f"Waiting for vLLM... ({i*5}s)")
        time.sleep(5)
else:
    print("ERROR: vLLM did not start within 5 minutes.")
    print("Check GPU memory — try reducing --max-model-len.")

## 3. Load a sample document

Upload your sample PDFs/TIFFs to `/ocr/` using Jupyter's file upload button.

In [ ]:
# List what's in /ocr
ocr_dir = Path("/ocr")
files = [f for f in ocr_dir.iterdir() if f.is_file()]
print("Files in /ocr/:")
for f in sorted(files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.0f} KB)")

if not files:
    print("\nNo files found. Upload your sample docs to /ocr/ first.")

In [ ]:
# Set this to one of your uploaded files
DOC_PATH = Path("/ocr/sample.pdf")  # <-- UPDATE THIS

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH.name} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Check digital vs scanned

Extract text from each page. Pages with text are digital (fast path).
Pages without text are scanned (VLM path).

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

page_info = []
for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    page_info.append({"page": i, "text": text, "has_text": has_text})
    status = "DIGITAL" if has_text else "SCANNED"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:150]}...")
    print()

digital = sum(1 for p in page_info if p["has_text"])
scanned = sum(1 for p in page_info if not p["has_text"])
print(f"Summary: {digital} digital, {scanned} scanned")
doc.close()

## 5. Run extraction

Pick a page and run the appropriate extraction path.

In [ ]:
import base64
import io
from PIL import Image

# Which page to test (0-indexed)
PAGE_IDX = 0
info = page_info[PAGE_IDX]

# Prompt — try different ones from step 7!
PROMPT = """Extract all information from this document.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

t0 = time.time()

if info["has_text"]:
    print(f"Page {PAGE_IDX+1}: Using DIGITAL path (text extraction + LLM)\n")
    full_prompt = f"{PROMPT}\n\n---\nDOCUMENT TEXT:\n---\n{info['text']}"
    resp = httpx.post(
        f"{LLM_BASE_URL}/chat/completions",
        json={"model": MODEL_ID, "messages": [{"role": "user", "content": full_prompt}],
              "max_tokens": 4096, "temperature": 0.0},
        timeout=120.0,
    )
else:
    print(f"Page {PAGE_IDX+1}: Using SCANNED path (VLM OCR)\n")
    doc = fitz.open(str(DOC_PATH))
    mat = fitz.Matrix(2.0, 2.0)
    pix = doc[PAGE_IDX].get_pixmap(matrix=mat)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    print(f"Rendered: {img.width}x{img.height}")
    display(img.resize((img.width // 3, img.height // 3)))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode()
    resp = httpx.post(
        f"{LLM_BASE_URL}/chat/completions",
        json={"model": MODEL_ID, "messages": [{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": PROMPT},
        ]}], "max_tokens": 4096, "temperature": 0.0},
        timeout=120.0,
    )

elapsed = time.time() - t0
result = resp.json()["choices"][0]["message"]["content"]
print(f"\nExtraction took {elapsed:.1f}s")

## 6. Inspect the output

In [ ]:
import json

try:
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Not valid JSON: {e}\n")
    print("Raw output:")
    print(result)

## 7. Try different prompts

Copy one of these into the `PROMPT` variable in step 5 and re-run.

In [ ]:
PROMPT_KV = """Extract all labeled data points from this document as key-value pairs.
Return a JSON object where keys are the field names and values are their values.
Preserve ALL values exactly. Output only valid JSON."""

PROMPT_BUDGET = """Extract budget information from this document.
Return a JSON object with: award_number, budget_period,
categories (array of {category, items: [{description, amount}], subtotal}),
total_direct, fa_rate, fa_base, total_indirect, total, cost_sharing, notes.
Preserve ALL dollar amounts exactly. Output only valid JSON."""

PROMPT_TERMS = """Extract terms and conditions from this document.
Return a JSON object with: document_title, effective_date,
sections (array of {number, title, text, subsections}),
definitions, references.
Preserve exact wording. Output only valid JSON."""

PROMPT_TEXT = """Extract all text from this document exactly as it appears.
Preserve the original reading order, line breaks, and structure.
Output only the extracted text."""

print("Copy one of these into PROMPT in step 5 and re-run that cell.")
print("Available: PROMPT_KV, PROMPT_BUDGET, PROMPT_TERMS, PROMPT_TEXT")

## 8. Process all pages

Once you're happy with a prompt, run it on all pages.

In [ ]:
results = []
doc = fitz.open(str(DOC_PATH))

for info in page_info:
    t0 = time.time()
    if info["has_text"]:
        full_prompt = f"{PROMPT}\n\n---\nDOCUMENT TEXT:\n---\n{info['text']}"
        resp = httpx.post(
            f"{LLM_BASE_URL}/chat/completions",
            json={"model": MODEL_ID, "messages": [{"role": "user", "content": full_prompt}],
                  "max_tokens": 4096, "temperature": 0.0},
            timeout=120.0)
        method = "text_extraction"
    else:
        mat = fitz.Matrix(2.0, 2.0)
        pix = doc[info["page"]].get_pixmap(matrix=mat)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        resp = httpx.post(
            f"{LLM_BASE_URL}/chat/completions",
            json={"model": MODEL_ID, "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": PROMPT},
            ]}], "max_tokens": 4096, "temperature": 0.0},
            timeout=120.0)
        method = "vlm_ocr"
    elapsed = time.time() - t0
    text = resp.json()["choices"][0]["message"]["content"]
    results.append({"page": info["page"] + 1, "method": method,
                    "elapsed_ms": round(elapsed * 1000, 1), "text": text})
    print(f"Page {info['page']+1}: {method} ({elapsed:.1f}s)")

doc.close()
print(f"\nDone. {len(results)} pages processed.")

In [ ]:
# Save results as JSON
output = {
    "source_file": str(DOC_PATH),
    "total_pages": len(results),
    "digital_pages": sum(1 for r in results if r["method"] == "text_extraction"),
    "scanned_pages": sum(1 for r in results if r["method"] == "vlm_ocr"),
    "pages": results,
}
out_path = Path(f"/ocr/{DOC_PATH.stem}_extracted.json")
out_path.write_text(json.dumps(output, indent=2))
print(f"Saved to {out_path}")

## 9. Test Streamlit app

Launch the extraction server and Streamlit UI from this notebook.
Access at: `https://<cluster-host>/<project>/ocr-setup/proxy/8501/`

**Requires a Custom URL tool on port 8501** configured in the workspace
(in addition to Jupyter on 8888).

In [ ]:
import subprocess

# Start extraction server (background)
env = os.environ.copy()
env["LLM_BASE_URL"] = LLM_BASE_URL
env["VLM_MODEL"] = MODEL_ID

extract_proc = subprocess.Popen(
    ["python", "/tmp/KohakuRAG_UI/ocr_app/scripts/ocr_server.py"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"Extraction server starting (PID {extract_proc.pid})...")
time.sleep(3)

# Start Streamlit (background)
env["OCR_SERVICE_URL"] = "http://localhost:8090"
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "/tmp/KohakuRAG_UI/ocr_app/app.py",
     "--server.port=8501", "--server.address=0.0.0.0",
     "--server.headless=true"],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(f"Streamlit starting (PID {streamlit_proc.pid})...")
print(f"\nAccess at: https://<cluster-host>/<project>/ocr-setup/proxy/8501/")

In [ ]:
# Run this cell to stop Streamlit and extraction server
extract_proc.terminate()
streamlit_proc.terminate()
print("Stopped.")

## 10. Cleanup

Stop vLLM when done.

In [ ]:
vllm_proc.terminate()
print("vLLM stopped.")